# Investigation: Exact Building ID Matching for Centaline OIR

This notebook investigates whether we can use exact building IDs instead of fuzzy matching for joining transactions with building details.

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## Step 1: Load Current Data
Load transaction and building details data to see what IDs are currently available.

In [ ]:
# Load transaction data
trans_file = 'data/01_raw/centaline_oir_trans_lv_0.parquet'
if Path(trans_file).exists():
    df_trans = pd.read_parquet(trans_file)
    print(f"✓ Loaded {len(df_trans):,} transaction records")
    print(f"\nTransaction columns ({len(df_trans.columns)}):")
    print(df_trans.columns.tolist())
else:
    print(f"✗ File not found: {trans_file}")
    df_trans = None

In [ ]:
# Load building details
building_file = 'data/02_intermediate/centanet_oir_details.parquet'
if Path(building_file).exists():
    df_buildings = pd.read_parquet(building_file)
    print(f"✓ Loaded {len(df_buildings):,} building records")
    print(f"\nBuilding columns ({len(df_buildings.columns)}):")
    print(df_buildings.columns.tolist())
else:
    print(f"✗ File not found: {building_file}")
    df_buildings = None

In [ ]:
# Load building listings (intermediate data with propertyID)
listings_file = 'data/01_raw/centanet_oir_buildings.parquet'
if Path(listings_file).exists():
    df_listings = pd.read_parquet(listings_file)
    print(f"✓ Loaded {len(df_listings):,} building listing records")
    print(f"\nListing columns ({len(df_listings.columns)}):")
    print(df_listings.columns.tolist())
    print(f"\nSample data:")
    display(df_listings.head(3))
else:
    print(f"✗ File not found: {listings_file}")
    df_listings = None

## Step 2: Check Property ID Availability
Investigate if propertyId exists in both datasets and how well populated it is.

In [ ]:
if df_trans is not None:
    print("=" * 80)
    print("TRANSACTION DATA - Property ID Analysis")
    print("=" * 80)
    
    # Check for ID columns
    id_cols = [col for col in df_trans.columns if 'id' in col.lower() or 'property' in col.lower()]
    print(f"\nColumns containing 'id' or 'property': {id_cols}")
    
    # Check propertyId specifically
    if 'propertyId' in df_trans.columns:
        total = len(df_trans)
        non_null = df_trans['propertyId'].notna().sum()
        null_count = df_trans['propertyId'].isna().sum()
        unique_count = df_trans['propertyId'].nunique()
        
        print(f"\n✓ 'propertyId' column EXISTS in transaction data")
        print(f"  - Total records: {total:,}")
        print(f"  - Non-null values: {non_null:,} ({non_null/total*100:.2f}%)")
        print(f"  - Null values: {null_count:,} ({null_count/total*100:.2f}%)")
        print(f"  - Unique property IDs: {unique_count:,}")
        
        # Show sample values
        print(f"\n  Sample propertyId values:")
        sample_ids = df_trans['propertyId'].dropna().head(10)
        for idx, pid in enumerate(sample_ids, 1):
            print(f"    {idx}. {pid}")
    else:
        print("\n✗ 'propertyId' column NOT FOUND in transaction data")
    
    # Check other potential ID fields
    for col in ['ibsBuildingID', 'centabldg', 'ibsPropertyNo']:
        if col in df_trans.columns:
            non_null = df_trans[col].notna().sum()
            print(f"\n  '{col}': {non_null:,}/{len(df_trans):,} populated ({non_null/len(df_trans)*100:.2f}%)")
            print(f"    Sample: {df_trans[col].dropna().head(3).tolist()}")

In [ ]:
if df_buildings is not None:
    print("=" * 80)
    print("BUILDING DETAILS - Property ID Analysis")
    print("=" * 80)
    
    # Check for ID columns
    id_cols = [col for col in df_buildings.columns if 'id' in col.lower() or 'property' in col.lower()]
    print(f"\nColumns containing 'id' or 'property': {id_cols}")
    
    # Check property_id specifically
    if 'property_id' in df_buildings.columns:
        total = len(df_buildings)
        non_null = df_buildings['property_id'].notna().sum()
        unique_count = df_buildings['property_id'].nunique()
        
        print(f"\n✓ 'property_id' column EXISTS in building details")
        print(f"  - Total records: {total:,}")
        print(f"  - Non-null values: {non_null:,} ({non_null/total*100:.2f}%)")
        print(f"  - Unique property IDs: {unique_count:,}")
        
        # Show sample values
        print(f"\n  Sample property_id values:")
        sample_ids = df_buildings['property_id'].dropna().head(10)
        for idx, pid in enumerate(sample_ids, 1):
            print(f"    {idx}. {pid}")
    else:
        print("\n✗ 'property_id' column NOT FOUND in building details")

## Step 3: Test Exact ID Matching vs Fuzzy Matching
Compare match rates between exact ID matching and fuzzy name matching.

In [ ]:
if df_trans is not None and df_buildings is not None:
    print("=" * 80)
    print("MATCHING COMPARISON: Exact ID vs Fuzzy Name")
    print("=" * 80)
    
    # Method 1: Exact ID matching
    if 'propertyId' in df_trans.columns and 'property_id' in df_buildings.columns:
        # Cast to string for comparison
        df_trans_test = df_trans.copy()
        df_buildings_test = df_buildings.copy()
        df_trans_test['propertyId'] = df_trans_test['propertyId'].astype(str)
        df_buildings_test['property_id'] = df_buildings_test['property_id'].astype(str)
        
        # Perform exact join
        exact_match = df_trans_test.merge(
            df_buildings_test,
            left_on='propertyId',
            right_on='property_id',
            how='left',
            indicator=True
        )
        
        total_trans = len(df_trans_test)
        matched_count = (exact_match['_merge'] == 'both').sum()
        unmatched_count = (exact_match['_merge'] == 'left_only').sum()
        
        print(f"\n📊 EXACT ID MATCHING RESULTS:")
        print(f"  - Total transactions: {total_trans:,}")
        print(f"  - Matched: {matched_count:,} ({matched_count/total_trans*100:.2f}%)")
        print(f"  - Unmatched: {unmatched_count:,} ({unmatched_count/total_trans*100:.2f}%)")
        
        # Show sample matched records
        print(f"\n  Sample matched records:")
        matched_sample = exact_match[exact_match['_merge'] == 'both'][['propertyId', 'propertyNameEn', 'building_name', 'property_id']].head(5)
        display(matched_sample)
        
        # Show sample unmatched records
        print(f"\n  Sample unmatched records (no building details found):")
        unmatched_sample = exact_match[exact_match['_merge'] == 'left_only'][['propertyId', 'propertyNameEn']].head(5)
        display(unmatched_sample)
    else:
        print("\n✗ Cannot perform exact ID matching - required columns missing")
    
    # Method 2: Check fuzzy matching results (if available)
    fuzzy_stats_file = 'data/08_reporting/fuzzy_match_stats.csv'
    if Path(fuzzy_stats_file).exists():
        df_fuzzy = pd.read_csv(fuzzy_stats_file)
        print(f"\n📊 FUZZY MATCHING RESULTS (from existing stats):")
        print(f"  - Total fuzzy matches: {len(df_fuzzy):,}")
        print(f"  - Average match score: {df_fuzzy['match_score'].mean():.2f}%")
        print(f"  - Min match score: {df_fuzzy['match_score'].min():.2f}%")
        print(f"  - Max match score: {df_fuzzy['match_score'].max():.2f}%")
        
        # Distribution of match quality
        print(f"\n  Match quality distribution:")
        print(f"    - Perfect match (100%): {(df_fuzzy['match_score'] == 100).sum():,}")
        print(f"    - Excellent (95-99%): {((df_fuzzy['match_score'] >= 95) & (df_fuzzy['match_score'] < 100)).sum():,}")
        print(f"    - Good (85-94%): {((df_fuzzy['match_score'] >= 85) & (df_fuzzy['match_score'] < 95)).sum():,}")
        print(f"    - Fair (80-84%): {((df_fuzzy['match_score'] >= 80) & (df_fuzzy['match_score'] < 85)).sum():,}")
        
        # Sample fuzzy matches
        print(f"\n  Sample fuzzy matches:")
        display(df_fuzzy[['transaction_name', 'matched_building_name', 'match_score']].head(10))

## Step 4: Investigate API Response Structure
Check what IDs are available in the API response by making a sample request.

In [ ]:
# Sample API request to understand structure
import time
import random

print("=" * 80)
print("API RESPONSE INVESTIGATION")
print("=" * 80)

# Transaction API sample
transaction_api_url = "https://oir.centanet.com/api/Transaction/GetTransactionList"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

# Make a small test request
params = {
    "pageindex": 1,
    "pagesize": 5,
    "daterang": "01/01/2025-30/01/2025",
    "sellpricetype": "TOTAL",
    "rentpricetype": "TOTAL",
    "districtids": "1",  # Central & Western
    "lang": "EN"
}

try:
    print("\n📡 Making test API request to Transaction API...")
    time.sleep(random.uniform(0.5, 1.5))  # Be polite
    response = requests.get(transaction_api_url, headers=headers, params=params, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        items = data.get("data", {}).get("recordList", {}).get("items", [])
        
        if items:
            print(f"\n✓ Successfully retrieved {len(items)} sample transactions")
            
            # Examine first item structure
            print("\n" + "="*80)
            print("SAMPLE TRANSACTION RECORD STRUCTURE:")
            print("="*80)
            sample_item = items[0]
            
            # Pretty print with indentation
            print(json.dumps(sample_item, indent=2, ensure_ascii=False))
            
            # Highlight key ID fields
            print("\n" + "="*80)
            print("KEY ID FIELDS IN TRANSACTION:")
            print("="*80)
            id_fields = {}
            for key, value in sample_item.items():
                if 'id' in key.lower() or 'property' in key.lower():
                    id_fields[key] = value
            
            print(json.dumps(id_fields, indent=2, ensure_ascii=False))
            
            # Check nested propertyUrlInfo
            if 'propertyUrlInfo' in sample_item:
                print("\n" + "="*80)
                print("propertyUrlInfo (contains propertyId):")
                print("="*80)
                print(json.dumps(sample_item['propertyUrlInfo'], indent=2, ensure_ascii=False))
        else:
            print("\n⚠️ No transaction records returned")
    else:
        print(f"\n✗ API request failed with status code: {response.status_code}")
        
except Exception as e:
    print(f"\n✗ Error making API request: {e}")

## Step 5: Analysis of Building Listings API
Check the building listings API to see what IDs are available.

In [ ]:
# Building/Property API sample
property_api_url = "https://oir.centanet.com/api/Property/GetPropertyList"

params = {
    "PageSize": 5,
    "pageindex": 1,
    "districtids": "1",  # Central & Western
    "lang": "EN"
}

try:
    print("\n📡 Making test API request to Property/Building API...")
    time.sleep(random.uniform(0.5, 1.5))  # Be polite
    response = requests.get(property_api_url, headers=headers, params=params, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        items = data.get("data", {}).get("items", [])
        
        if items:
            print(f"\n✓ Successfully retrieved {len(items)} sample buildings")
            
            # Examine first item structure
            print("\n" + "="*80)
            print("SAMPLE BUILDING RECORD STRUCTURE:")
            print("="*80)
            sample_building = items[0]
            
            # Pretty print
            print(json.dumps(sample_building, indent=2, ensure_ascii=False))
            
            # Highlight propertyID
            print("\n" + "="*80)
            print("KEY IDENTIFIER:")
            print("="*80)
            if 'propertyID' in sample_building:
                print(f"propertyID: {sample_building['propertyID']}")
                print(f"buildingNameEn: {sample_building.get('buildingNameEn', 'N/A')}")
        else:
            print("\n⚠️ No building records returned")
    else:
        print(f"\n✗ API request failed with status code: {response.status_code}")
        
except Exception as e:
    print(f"\n✗ Error making API request: {e}")

## Step 6: Check for Missing Links
Investigate why some transactions might not have propertyId.

In [ ]:
if df_trans is not None and 'propertyId' in df_trans.columns:
    print("=" * 80)
    print("INVESTIGATING MISSING PROPERTY IDs IN TRANSACTIONS")
    print("=" * 80)
    
    # Separate records with and without propertyId
    has_id = df_trans[df_trans['propertyId'].notna()]
    no_id = df_trans[df_trans['propertyId'].isna()]
    
    print(f"\nTransactions WITH propertyId: {len(has_id):,}")
    print(f"Transactions WITHOUT propertyId: {len(no_id):,}")
    
    if len(no_id) > 0:
        print("\n🔍 Sample transactions WITHOUT propertyId:")
        cols_to_show = ['propertyNameEn', 'propertyId', 'ibsBuildingID', 'centabldg', 'transactionDate', 'District']
        cols_available = [col for col in cols_to_show if col in df_trans.columns]
        display(no_id[cols_available].head(10))
        
        # Check if they have other IDs
        print("\n📊 Alternative ID availability for records WITHOUT propertyId:")
        for id_col in ['ibsBuildingID', 'centabldg', 'ibsPropertyNo']:
            if id_col in no_id.columns:
                filled = no_id[id_col].notna().sum()
                print(f"  - {id_col}: {filled:,}/{len(no_id):,} ({filled/len(no_id)*100:.2f}%)")
    
    if len(has_id) > 0:
        print("\n✓ Sample transactions WITH propertyId:")
        cols_to_show = ['propertyNameEn', 'propertyId', 'ibsBuildingID', 'transactionDate', 'District']
        cols_available = [col for col in cols_to_show if col in df_trans.columns]
        display(has_id[cols_available].head(10))

## Step 7: Recommendation Summary
Based on the analysis above, provide recommendations.

In [ ]:
print("="*80)
print("RECOMMENDATIONS FOR EXACT BUILDING ID MATCHING")
print("="*80)

if df_trans is not None and 'propertyId' in df_trans.columns and df_buildings is not None and 'property_id' in df_buildings.columns:
    # Calculate metrics
    trans_with_id = df_trans['propertyId'].notna().sum()
    trans_total = len(df_trans)
    coverage = trans_with_id / trans_total * 100
    
    print(f"\n📊 Current Status:")
    print(f"  ✓ propertyId exists in transaction data ({coverage:.2f}% populated)")
    print(f"  ✓ property_id exists in building details")
    
    if coverage >= 95:
        print(f"\n✅ RECOMMENDATION: USE EXACT ID MATCHING")
        print(f"  - Coverage is excellent ({coverage:.2f}%)")
        print(f"  - Exact matching will be more reliable than fuzzy matching")
        print(f"  - No need to scrape additional data")
    elif coverage >= 80:
        print(f"\n⚠️ RECOMMENDATION: USE HYBRID APPROACH")
        print(f"  - Coverage is good but not complete ({coverage:.2f}%)")
        print(f"  - Use exact ID matching as primary method")
        print(f"  - Use fuzzy matching as fallback for records without propertyId")
    else:
        print(f"\n❌ RECOMMENDATION: INVESTIGATE WHY propertyId COVERAGE IS LOW")
        print(f"  - Coverage is too low ({coverage:.2f}%)")
        print(f"  - Check if API response includes propertyId")
        print(f"  - May need to enhance scraping logic")
    
    print(f"\n📝 Implementation Notes:")
    print(f"  1. propertyId is already being scraped from 'propertyUrlInfo' in transactions")
    print(f"  2. propertyID is available in building listings API")
    print(f"  3. No duplicate scraping needed - data is already available")
    print(f"  4. Simply change join configuration to use exact matching:")
    print(f"")
    print(f"     In parameters.yml:")
    print(f"     centaline_oir:")
    print(f"       join:")
    print(f"         use_fuzzy_matching: false  # <-- Change this to false")
    print(f"         left_on: 'propertyId'")
    print(f"         right_on: 'property_id'")
    print(f"         type: 'left'")
else:
    print(f"\n❌ Cannot provide recommendation - required data not available")
    print(f"\nPlease ensure:")
    print(f"  1. Transaction data is loaded")
    print(f"  2. Building details data is loaded")
    print(f"  3. Both datasets have the required ID columns")